# Convert GTEP Solution to Prescient PCM Input (2035)

This notebook converts the solved GTEP model output (stage 3 = 2035) into
Prescient-compatible input data for the 123-bus ERCOT coal retirement case.

**Key corrections vs. the earlier `output_to_prescient.ipynb` (jkskolf):**
- Fuel prices are set from `fuel_cost3` (2035 projection), NOT hardcoded to 1
- `HR_incr_1` is the actual heat rate increment, NOT `fuel_cost3`
- Candidate CT heat rates derived from existing CT median
- Renewable PMax MW updated from GTEP continuous solution

**Scenario:** `retirement_allowed_no_extreme_half_load`, Stage 3 (2035)

In [1]:
import pandas as pd
import json
import re
import shutil
from pathlib import Path
from datetime import datetime

# --- CONFIGURATION ---
SCENARIO = "retirement_allowed_no_extreme_half_load_local"
GTEP_STAGE = 3  # 2035
DEV_ROOT = Path("/Users/yilu/Documents/development/nd/research/gtep/123_bus_coal")
REPO_ROOT = Path("/Users/yilu/Documents/GitHub/idaes-gtep")

# Input paths — GTEP solution is in non-_local dev directory (same solution for both)
SOLUTION_DIR = DEV_ROOT / "retirement_allowed_no_extreme_half_load"
TIMESERIES_DIR = SOLUTION_DIR / "Prescient"

# Existing generator sources:
#   Prescient_2/gen.csv — 292 rows, CORRECT cost curves (4-segment HR, real fuel prices, ramp rates)
#                         This is the file used by the actual 2019 simulations.
#   Prescient/gen.csv  — 374 rows, WRONG cost curves (Fuel Price=1, HR_incr=fuel_cost, 2-segment)
#                         This was jkskolf's intermediate file, NOT used for simulations.
#                         But it HAS fuel_cost3 column needed for 2035 pricing.
CORRECT_GEN = REPO_ROOT / f"gtep/data/{SCENARIO}/Prescient_2/gen.csv"  # 292 rows, correct HR/costs
FUEL_COST_GEN = REPO_ROOT / "gtep/data/123_Bus_Coal/Prescient/gen.csv"  # 374 rows, has fuel_cost3
CANDIDATE_GEN = REPO_ROOT / "gtep/data/123_Bus_Coal/candidate_generators_initial_list.csv"  # 127 candidates
BASE_POINTERS = REPO_ROOT / "gtep/data/123_Bus_Coal/timeseries_pointers.csv"  # 1206 rows

# Output path
OUTPUT_DIR = REPO_ROOT / f"gtep/data/{SCENARIO}/Prescient_2_2035"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Scenario: {SCENARIO}")
print(f"GTEP Stage: {GTEP_STAGE} (2035)")
print(f"Solution dir: {SOLUTION_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Correct gen source: {CORRECT_GEN}")
print(f"Fuel cost source: {FUEL_COST_GEN}")

Scenario: retirement_allowed_no_extreme_half_load_local
GTEP Stage: 3 (2035)
Solution dir: /Users/yilu/Documents/development/nd/research/gtep/123_bus_coal/retirement_allowed_no_extreme_half_load
Output: /Users/yilu/Documents/GitHub/idaes-gtep/gtep/data/retirement_allowed_no_extreme_half_load_local/Prescient_2_2035
Correct gen source: /Users/yilu/Documents/GitHub/idaes-gtep/gtep/data/retirement_allowed_no_extreme_half_load_local/Prescient_2/gen.csv
Fuel cost source: /Users/yilu/Documents/GitHub/idaes-gtep/gtep/data/123_Bus_Coal/Prescient/gen.csv


## Cell 2: Parse GTEP Solution (Stage 3)

The GTEP solution has two files:
- `dispatchable_investments.json` — binary (true) decisions for thermal generators
- `renewable_investments.json` — continuous MW capacity for renewables

Keys follow `investmentStage[N].<decision>.<gen_id>` format.

In [2]:
# --- Load GTEP solution files ---
with open(SOLUTION_DIR / "dispatchable_investments.json") as f:
    disp_raw = json.load(f)
with open(SOLUTION_DIR / "renewable_investments.json") as f:
    renew_raw = json.load(f)

stage_prefix = f"investmentStage[{GTEP_STAGE}]"

# --- Parse dispatchable (thermal) investments ---
# Filter to stage 3, extract decision type and gen ID
disp_stage3 = {}
for key, val in disp_raw.items():
    if not key.startswith(stage_prefix):
        continue
    parts = key.split(".")
    decision = parts[1]  # genOperational, genInstalled, genRetired, genDisabled, genExtended
    gen_id = parts[2]
    disp_stage3.setdefault(decision, {})[gen_id] = val

# Active thermal = Operational + Installed + Extended (all have value=True)
active_thermal = set()
for decision in ["genOperational", "genInstalled", "genExtended"]:
    active_thermal |= set(disp_stage3.get(decision, {}).keys())

retired_thermal = set(disp_stage3.get("genRetired", {}).keys())
disabled_thermal = set(disp_stage3.get("genDisabled", {}).keys())

# Remove retired/disabled from active set (shouldn't overlap, but be safe)
active_thermal -= retired_thermal
active_thermal -= disabled_thermal

print(f"Stage 3 thermal generators:")
print(f"  Active (operational/installed/extended): {len(active_thermal)}")
print(f"  Retired: {len(retired_thermal)}")
print(f"  Disabled (not built): {len(disabled_thermal)}")

# Classify active thermal by type
active_thermal_existing = {g for g in active_thermal if not g.endswith("-c")}
active_thermal_candidate = {g for g in active_thermal if g.endswith("-c")}
print(f"  Existing thermal kept: {len(active_thermal_existing)}")
print(f"  Candidate thermal built: {len(active_thermal_candidate)}")
print(f"  Candidate thermal list: {sorted(active_thermal_candidate)}")

# --- Parse renewable investments ---
renew_stage3 = {}
for key, val in renew_raw.items():
    if not key.startswith(stage_prefix):
        continue
    parts = key.split(".")
    decision = parts[1]
    gen_id = parts[2]
    renew_stage3.setdefault(decision, {})[gen_id] = val

# Renewable capacity = sum of Operational + Installed + Extended MW
renewable_capacity = {}
for decision in ["renewableOperational", "renewableInstalled", "renewableExtended"]:
    for gen_id, mw in renew_stage3.get(decision, {}).items():
        renewable_capacity[gen_id] = renewable_capacity.get(gen_id, 0) + mw

# Remove any with effectively zero capacity
renewable_capacity = {k: v for k, v in renewable_capacity.items() if v > 0.01}

renew_existing = {k: v for k, v in renewable_capacity.items() if not k.endswith("-c")}
renew_candidate = {k: v for k, v in renewable_capacity.items() if k.endswith("-c")}

print(f"\nStage 3 renewable generators:")
print(f"  Total with capacity: {len(renewable_capacity)}")
print(f"  Existing renewables: {len(renew_existing)}")
print(f"  Candidate renewables: {len(renew_candidate)}")
print(f"  Total installed renewable MW: {sum(renewable_capacity.values()):.1f}")

# Full set of invested generators
invested_gen_set = active_thermal | set(renewable_capacity.keys())
print(f"\nTotal invested generators: {len(invested_gen_set)}")

Stage 3 thermal generators:
  Active (operational/installed/extended): 149
  Retired: 0
  Disabled (not built): 96
  Existing thermal kept: 127
  Candidate thermal built: 22
  Candidate thermal list: ['ct_fe_107_2-c', 'ct_fe_111_2-c', 'ct_fe_111_3-c', 'ct_fe_112_1-c', 'ct_fe_112_3-c', 'ct_fe_113_3-c', 'ct_fe_118_3-c', 'ct_fe_19_1-c', 'ct_fe_19_3-c', 'ct_fe_32_1-c', 'ct_fe_33_2-c', 'ct_fe_35_3-c', 'ct_fe_49_1-c', 'ct_fe_49_2-c', 'ct_fe_58_3-c', 'ct_fe_61_2-c', 'ct_fe_61_3-c', 'ct_fe_63_1-c', 'ct_fe_65_2-c', 'ct_fe_66_1-c', 'ct_fe_8_1-c', 'ct_fe_98_1-c']

Stage 3 renewable generators:
  Total with capacity: 129
  Existing renewables: 68
  Candidate renewables: 61
  Total installed renewable MW: 11268.1

Total invested generators: 278


## Cell 3: Build Merged gen.csv

Strategy:
1. Start from `Prescient_2/gen.csv` (292 existing gens with CORRECT 4-segment cost curves, ramp rates)
2. Merge `fuel_cost3` from `Prescient/gen.csv` for 2035 fuel pricing
3. Add candidate CTs and renewable candidates from `candidate_generators_initial_list.csv`
4. Synthesize generators that exist in GTEP solution but not in any gen.csv
5. **Back-calculate Fuel Price**: `fuel_cost3` is $/MWh (see `gtep_model.py:1889`), so
   `Fuel Price $/MMBTU = fuel_cost3 / (HR_incr_1 × 0.001)` to ensure MC = fuel_cost3
6. Fill missing HR/startup for candidate CTs using existing CT medians
7. Force `Output_pct_0 = 0` for all renewables (curtailable to zero)
8. Update renewable PMax MW from GTEP solution

**CRITICAL FIX (2026-04-02):** `fuel_cost3` has units `USD/(MW·hr)` = $/MWh (total marginal cost),
NOT $/MMBTU. Previous version placed it directly in "Fuel Price $/MMBTU", causing Prescient to
compute `MC = fuel_cost3 × HR_incr × 0.001` — a double-counting that inflated COAL costs by 9×
and NUC costs by 16×. CTs appeared correct by coincidence (HR ≈ 1000, so ×0.001 ≈ 1.0).

**CRITICAL FIX (2026-04-01):** Uses `Prescient_2/gen.csv` (correct cost curves) NOT `Prescient/gen.csv`
(which had `Fuel Price=1` and `HR_incr=fuel_cost` — jkskolf's encoding error).

In [3]:
# --- Read source data ---
# 1. Prescient_2/gen.csv: correct 4-segment cost curves, ramp rates, fuel prices (2019)
correct_gen = pd.read_csv(CORRECT_GEN)
correct_gen["GEN UID"] = correct_gen["GEN UID"].astype(str)
print(f"Prescient_2/gen.csv (correct curves): {len(correct_gen)} rows")
print(f"  Columns: {len(correct_gen.columns)}")
print(f"  Has Ramp Rate: {correct_gen['Ramp Rate MW/Min'].notna().sum()} non-null")
print(f"  Has HR_incr_2: {correct_gen['HR_incr_2'].notna().sum()} non-null")

# 2. Prescient/gen.csv: has fuel_cost3 for 2035 pricing (but wrong HR values — only use fuel_cost3)
fuel_cost_gen = pd.read_csv(FUEL_COST_GEN)
fuel_cost_gen["GEN UID"] = fuel_cost_gen["GEN UID"].astype(str)
fuel_cost_lookup = fuel_cost_gen.set_index("GEN UID")["fuel_cost3"].to_dict()
print(f"\nFuel cost source: {len(fuel_cost_gen)} rows, {len(fuel_cost_lookup)} fuel_cost3 values")

# 3. Candidate generators (CT + PV + WIND)
candidate_gen = pd.read_csv(CANDIDATE_GEN)
candidate_gen["GEN UID"] = candidate_gen["GEN UID"].astype(str)
print(f"Candidate gen: {len(candidate_gen)} rows")

# --- Compute CT median heat rates from existing generators ---
existing_ct = correct_gen[correct_gen["Unit Type"] == "CT"]
ct_median_hr_incr_1 = existing_ct["HR_incr_1"].median()
ct_median_hr_incr_2 = existing_ct["HR_incr_2"].median()
ct_median_hr_incr_3 = existing_ct["HR_incr_3"].median()
ct_median_ramp = existing_ct["Ramp Rate MW/Min"].median()
ct_median_csu = existing_ct["Csu"].median()
print(f"\nExisting CT medians (from Prescient_2/gen.csv):")
print(f"  HR_incr_1: {ct_median_hr_incr_1:.2f}")
print(f"  HR_incr_2: {ct_median_hr_incr_2:.2f}")
print(f"  HR_incr_3: {ct_median_hr_incr_3:.2f}")
print(f"  Ramp Rate: {ct_median_ramp:.2f} MW/Min")
print(f"  Csu:       {ct_median_csu:.2f}")

# --- Build combined gen DataFrame ---
# Start with correct_gen (existing generators with proper cost curves)
# Then add candidate generators from candidate_gen
# Align candidate columns to correct_gen schema
target_cols = correct_gen.columns.tolist()

# Prepare candidate rows
cand_rows = candidate_gen.copy()
for col in target_cols:
    if col not in cand_rows.columns:
        cand_rows[col] = pd.NA
# Drop GTEP-only columns from candidates (will be dropped at end anyway)
combined_gen = pd.concat([correct_gen, cand_rows[target_cols]], ignore_index=True)

# --- Synthesize entries for candidates not in any gen.csv ---
invested_gen_str = {str(g) for g in invested_gen_set}
combined_uids = set(combined_gen["GEN UID"])
still_missing = invested_gen_str - combined_uids

if still_missing:
    print(f"\nSynthesizing {len(still_missing)} generators not in any source gen.csv")
    # Templates: use a representative existing generator of the same type
    ct_template = correct_gen[correct_gen["Unit Type"] == "CT"].iloc[0].copy()
    wind_template = correct_gen[correct_gen["Unit Type"] == "WIND"].iloc[0].copy()
    pv_template = correct_gen[correct_gen["Unit Type"] == "PV"].iloc[0].copy()

    synth_rows = []
    for gen_id in sorted(still_missing):
        if gen_id.startswith("ct_fe_"):
            row = ct_template.copy()
            bus_id = int(gen_id.split("_")[2])
        elif gen_id.startswith("pv_"):
            row = pv_template.copy()
            bus_id = int(gen_id.replace("pv_", "").replace("-c", ""))
        elif gen_id.startswith("wind_"):
            row = wind_template.copy()
            bus_id = int(gen_id.replace("wind_", "").replace("-c", ""))
        else:
            print(f"  Skipping unknown type: {gen_id}")
            continue

        row["GEN UID"] = gen_id
        row["Bus ID"] = bus_id
        if gen_id in renewable_capacity:
            row["PMax MW"] = renewable_capacity[gen_id]
        synth_rows.append(row)
        print(f"  Synthesized: {gen_id} (Bus {bus_id})")

    synth_df = pd.DataFrame(synth_rows)
    combined_gen = pd.concat([combined_gen, synth_df], ignore_index=True)

print(f"\nCombined gen.csv: {len(combined_gen)} rows")

# --- Filter to invested generators ---
gen_df = combined_gen[combined_gen["GEN UID"].isin(invested_gen_str)].copy()
print(f"Filtered to invested generators: {len(gen_df)} rows")

found = set(gen_df["GEN UID"])
missing_final = invested_gen_str - found
if missing_final:
    print(f"WARNING: {len(missing_final)} still missing: {sorted(missing_final)}")
else:
    print("All invested generators found.")

# --- Merge fuel_cost3 for 2035 pricing ---
gen_df["fuel_cost3"] = gen_df["GEN UID"].map(fuel_cost_lookup)
# For candidates from candidate_gen, also check their fuel_cost3
cand_fuel = candidate_gen.set_index("GEN UID")["fuel_cost3"].to_dict()
gen_df["fuel_cost3"] = gen_df["fuel_cost3"].fillna(gen_df["GEN UID"].map(cand_fuel))

# Derive fuel_cost3 by fuel type for any remaining NaN
fuel_type_defaults = {
    "G": 22.80128,   # Gas CT
    "C": 18.93942,   # Coal
    "N": 7.378403,   # Nuclear
    "W": 0.0,        # Wind
    "S": 0.0,        # Solar
    "H": 0.0,        # Hydro
}
for fuel_code, default_cost in fuel_type_defaults.items():
    mask = gen_df["fuel_cost3"].isna() & (gen_df["Fuel"] == fuel_code)
    if mask.any():
        gen_df.loc[mask, "fuel_cost3"] = default_cost
        print(f"  Filled fuel_cost3={default_cost} for {mask.sum()} generators (Fuel={fuel_code})")

# --- Fill missing columns for candidate CTs (BEFORE fuel price calc, so HR_incr_1 is populated) ---
is_candidate_ct = gen_df["GEN UID"].str.startswith("ct_fe_")
gen_df.loc[is_candidate_ct & gen_df["HR_avg_0"].isna(), "HR_avg_0"] = 0
gen_df.loc[is_candidate_ct & gen_df["HR_incr_1"].isna(), "HR_incr_1"] = ct_median_hr_incr_1
gen_df.loc[is_candidate_ct & gen_df["HR_incr_2"].isna(), "HR_incr_2"] = ct_median_hr_incr_2
gen_df.loc[is_candidate_ct & gen_df["HR_incr_3"].isna(), "HR_incr_3"] = ct_median_hr_incr_3
# Give candidate CTs the standard 4-segment output points
gen_df.loc[is_candidate_ct & gen_df["Output_pct_2"].isna(), "Output_pct_1"] = 0.533333333
gen_df.loc[is_candidate_ct & gen_df["Output_pct_2"].isna(), "Output_pct_2"] = 0.766666667
gen_df.loc[is_candidate_ct & gen_df["Output_pct_3"].isna(), "Output_pct_3"] = 1.0
for col in ["Start Time Cold Hr", "Start Time Warm Hr", "Start Time Hot Hr"]:
    gen_df.loc[is_candidate_ct & gen_df[col].isna(), col] = 1
for col in ["Start Heat Cold MBTU", "Start Heat Warm MBTU", "Start Heat Hot MBTU"]:
    gen_df.loc[is_candidate_ct & gen_df[col].isna(), col] = 1
gen_df.loc[is_candidate_ct & gen_df["Ramp Rate MW/Min"].isna(), "Ramp Rate MW/Min"] = ct_median_ramp
if "Csu" in gen_df.columns:
    gen_df.loc[is_candidate_ct & gen_df["Csu"].isna(), "Csu"] = ct_median_csu
if "Non Fuel Start Cost $" in gen_df.columns:
    gen_df.loc[is_candidate_ct & gen_df["Non Fuel Start Cost $"].isna(), "Non Fuel Start Cost $"] = 0

# ===========================================================================
# FIX: Recompute HR_avg_0 and HR_incr from C0/C1/C2 cost curves
# ===========================================================================
# Source gen.csv has corrupted HR values from demo_processing_prescient.ipynb:
#   HR_avg_0 = Csu*PMax/FP (startup cost formula, NOT a heat rate)
#   HR_incr = delta_cost/FP (total segment fuel, NOT per-MW incremental rate)
# Correct: derive proper BTU/kWh heat rates from C(P) = C0 + C1*P + C2*P^2.
# Egret parser (parser.py:574) uses HR_avg_0 to compute PMin fuel cost:
#   f[0] = (HR_avg_0 * 1000 / 1e6) * PMin  →  fuel_cost = FP * f[0]
# Corrupted HR_avg_0 of 3.6M for NUC gives $1.2M/hr PMin cost → never commits.

def recompute_hr_from_cost_curve(gen_df):
    """Recompute HR_avg_0 and HR_incr_1/2/3 from C0/C1/C2 cost curves."""
    thermal_with_cost = (
        gen_df["C0"].notna() & gen_df["C1"].notna() & gen_df["C2"].notna()
        & (gen_df["Fuel Price $/MMBTU"].astype(float) > 0.001)
        & ~gen_df["Unit Type"].isin(["WIND", "PV", "HYDRO"])
    )

    for idx in gen_df[thermal_with_cost].index:
        c0 = float(gen_df.at[idx, "C0"])
        c1 = float(gen_df.at[idx, "C1"])
        c2 = float(gen_df.at[idx, "C2"])
        fp = float(gen_df.at[idx, "Fuel Price $/MMBTU"])
        pmax = float(gen_df.at[idx, "PMax MW"])

        # Get output breakpoints (MW)
        pcts = []
        for i in range(4):
            col = f"Output_pct_{i}"
            if col in gen_df.columns and pd.notna(gen_df.at[idx, col]) and gen_df.at[idx, col] != "":
                pcts.append(float(gen_df.at[idx, col]))
            else:
                break
        if len(pcts) < 2 or fp < 0.001 or pmax <= 0:
            continue

        p = [pct * pmax for pct in pcts]  # MW breakpoints
        costs = [c0 + c1 * pi + c2 * pi**2 for pi in p]

        # HR_avg_0 = average heat rate at PMin (BTU/kWh)
        if p[0] > 0:
            gen_df.at[idx, "HR_avg_0"] = costs[0] / (fp * p[0]) * 1000

        # HR_incr_i = incremental heat rate for segment i (BTU/kWh)
        for i in range(1, len(p)):
            dp = p[i] - p[i-1]
            dc = costs[i] - costs[i-1]
            if dp > 0:
                gen_df.at[idx, f"HR_incr_{i}"] = dc / (fp * dp) * 1000

    return gen_df

gen_df = recompute_hr_from_cost_curve(gen_df)

print("\n=== HR Recomputation Results ===")
for ut in ["NUC", "COAL", "CT"]:
    sub = gen_df[gen_df["Unit Type"] == ut]
    if len(sub) > 0:
        hr0 = sub["HR_avg_0"].astype(float)
        hr1 = sub["HR_incr_1"].astype(float)
        print(f"  {ut}: HR_avg_0=[{hr0.min():.0f}, {hr0.max():.0f}], "
              f"HR_incr_1=[{hr1.min():.0f}, {hr1.max():.0f}]")

# --- Set 2035 fuel prices via BACK-CALCULATION from fuel_cost3 ---
# CRITICAL FIX (2026-04-02): fuel_cost3 is $/MWh (USD/(MW·hr)), NOT $/MMBTU.
# Prescient computes: MC = Fuel_Price × HR_incr_1 × 0.001
# Therefore: Fuel_Price = fuel_cost3 / (HR_incr_1 × 0.001)
# This ensures the first-segment MC exactly equals fuel_cost3 for each generator.
is_renewable = gen_df["Unit Type"].isin(["WIND", "PV", "HYDRO"])
gen_df["Fuel Price $/MMBTU"] = gen_df["Fuel Price $/MMBTU"].astype(float)

# Back-calculate Fuel Price for each thermal generator individually
thermal_mask = ~is_renewable & (gen_df["HR_incr_1"] > 0)
gen_df.loc[thermal_mask, "Fuel Price $/MMBTU"] = (
    gen_df.loc[thermal_mask, "fuel_cost3"].astype(float)
    / (gen_df.loc[thermal_mask, "HR_incr_1"].astype(float) * 0.001)
)
gen_df.loc[is_renewable, "Fuel Price $/MMBTU"] = 0.00001

print(f"\nFuel prices set (2035) — back-calculated from fuel_cost3 / (HR_incr_1 × 0.001):")
for ut in sorted(gen_df["Unit Type"].unique()):
    sub = gen_df[gen_df["Unit Type"] == ut]
    fp_min = sub["Fuel Price $/MMBTU"].min()
    fp_max = sub["Fuel Price $/MMBTU"].max()
    if fp_min == fp_max:
        print(f"  {ut}: Fuel Price = {fp_min:.5f} $/MMBTU (n={len(sub)})")
    else:
        print(f"  {ut}: Fuel Price = {fp_min:.5f}–{fp_max:.5f} $/MMBTU (n={len(sub)})")

# --- PMin Cost Sanity Check ---
# Verify PMin fuel cost is reasonable (not millions from corrupted HR_avg_0)
print("\n=== PMin Cost Sanity Check ===")
for ut in ["NUC", "COAL", "CT"]:
    sub = gen_df[gen_df["Unit Type"] == ut]
    if len(sub) > 0:
        pmin_cost = (sub["Fuel Price $/MMBTU"].astype(float)
                     * sub["HR_avg_0"].astype(float)
                     * sub["PMin MW"].astype(float) / 1000)
        print(f"  {ut}: PMin cost/hr range=[${pmin_cost.min():,.0f}, ${pmin_cost.max():,.0f}]")
        if pmin_cost.max() > 100_000:
            print(f"    WARNING: PMin cost exceeds $100K/hr — unit will never commit!")

# --- Fill missing columns for renewables ---
is_candidate_renew = gen_df["GEN UID"].str.match(r"^(pv|wind)_")
all_renew = is_renewable | is_candidate_renew
for col in ["HR_avg_0", "HR_incr_1", "HR_incr_2", "HR_incr_3", "HR_incr_4"]:
    if col in gen_df.columns:
        gen_df.loc[all_renew, col] = gen_df.loc[all_renew, col].fillna(0)
for col in ["Start Time Cold Hr", "Start Time Warm Hr", "Start Time Hot Hr",
            "Start Heat Cold MBTU", "Start Heat Warm MBTU", "Start Heat Hot MBTU",
            "Min Down Time Hr", "Min Up Time Hr"]:
    if col in gen_df.columns:
        gen_df.loc[is_candidate_renew & gen_df[col].isna(), col] = 0
if "Non Fuel Start Cost $" in gen_df.columns:
    gen_df.loc[is_candidate_renew & gen_df["Non Fuel Start Cost $"].isna(), "Non Fuel Start Cost $"] = 0
gen_df.loc[is_candidate_renew & gen_df["PMin MW"].isna(), "PMin MW"] = 0

# FIX: Force Output_pct_0 = 0 for ALL renewables (must be curtailable to zero)
gen_df.loc[all_renew, "Output_pct_0"] = 0.0

# --- Update renewable PMax MW from GTEP solution ---
for gen_id, mw in renewable_capacity.items():
    mask = gen_df["GEN UID"] == str(gen_id)
    if mask.any():
        gen_df.loc[mask, "PMax MW"] = mw

# --- Ensure Bus ID is clean integer ---
gen_df["Bus ID"] = gen_df["Bus ID"].astype(int)

# --- Drop GTEP-specific columns ---
gtep_cols = ["capex1", "capex2", "capex3", "fuel_cost1", "fuel_cost2", "fuel_cost3",
             "fixed_ops1", "fixed_ops2", "fixed_ops3", "var_ops1", "var_ops2", "var_ops3"]
gen_df = gen_df.drop(columns=[c for c in gtep_cols if c in gen_df.columns])

# Ensure BUS ID column
if "BUS ID" not in gen_df.columns:
    gen_df["BUS ID"] = gen_df["Bus ID"]

# --- Write output ---
gen_output_path = OUTPUT_DIR / "gen.csv"
gen_df.to_csv(gen_output_path, index=False)
print(f"\nWrote {len(gen_df)} generators to {gen_output_path}")

print(f"\nGenerator counts by Unit Type:")
print(gen_df["Unit Type"].value_counts().to_string())
print(f"\nTotal installed renewable MW: {gen_df.loc[all_renew, 'PMax MW'].sum():.1f}")

# --- Sanity check: marginal cost for sample generators ---
# Verify MC = Fuel_Price × HR_incr_1 × 0.001 ≈ fuel_cost3 for each fuel type
print(f"\n=== Cost Curve Sanity Check (MC should equal fuel_cost3) ===")
for uid, label in [("2", "CT"), ("26", "COAL"), ("1", "NUC")]:
    row = gen_df[gen_df["GEN UID"] == uid]
    if len(row) > 0:
        r = row.iloc[0]
        mc = r["Fuel Price $/MMBTU"] * r["HR_incr_1"] * 0.001
        print(f"  Gen {uid} ({label}): FuelPrice={r['Fuel Price $/MMBTU']:.5f} × HR_incr_1={r['HR_incr_1']:.2f} × 0.001 = ${mc:.2f}/MWh")

# Expected MC ranges by fuel type (from fuel_cost3)
expected_mc = {"CT": 22.80, "COAL": 18.94, "NUC": 7.38}
print(f"\n=== MC Validation by Fuel Type ===")
for ut, expected in expected_mc.items():
    sub = gen_df[gen_df["Unit Type"] == ut]
    if len(sub) > 0:
        mc_vals = sub["Fuel Price $/MMBTU"] * sub["HR_incr_1"] * 0.001
        mc_mean = mc_vals.mean()
        mc_min = mc_vals.min()
        mc_max = mc_vals.max()
        ok = "OK" if abs(mc_mean - expected) < 1.0 else "MISMATCH"
        print(f"  {ut}: MC mean=${mc_mean:.2f}, range=[${mc_min:.2f}, ${mc_max:.2f}], expected~${expected:.2f} {ok}")

Prescient_2/gen.csv (correct curves): 292 rows
  Columns: 36
  Has Ramp Rate: 292 non-null
  Has HR_incr_2: 292 non-null

Fuel cost source: 374 rows, 374 fuel_cost3 values
Candidate gen: 127 rows

Existing CT medians (from Prescient_2/gen.csv):
  HR_incr_1: 1360.40
  HR_incr_2: 1381.58
  HR_incr_3: 1402.77
  Ramp Rate: 45.25 MW/Min
  Csu:       44.00

Synthesizing 46 generators not in any source gen.csv
  Synthesized: ct_fe_107_2-c (Bus 107)
  Synthesized: ct_fe_111_2-c (Bus 111)
  Synthesized: ct_fe_111_3-c (Bus 111)
  Synthesized: ct_fe_112_1-c (Bus 112)
  Synthesized: ct_fe_112_3-c (Bus 112)
  Synthesized: ct_fe_113_3-c (Bus 113)
  Synthesized: ct_fe_118_3-c (Bus 118)
  Synthesized: ct_fe_19_1-c (Bus 19)
  Synthesized: ct_fe_19_3-c (Bus 19)
  Synthesized: ct_fe_32_1-c (Bus 32)
  Synthesized: ct_fe_33_2-c (Bus 33)
  Synthesized: ct_fe_35_3-c (Bus 35)
  Synthesized: ct_fe_49_1-c (Bus 49)
  Synthesized: ct_fe_49_2-c (Bus 49)
  Synthesized: ct_fe_58_3-c (Bus 58)
  Synthesized: ct_fe_61_

/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_54076/2235598805.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_gen = pd.concat([correct_gen, cand_rows[target_cols]], ignore_index=True)


## Cell 4: Build branch.csv

Copy the 2035 network branch data directly from the development directory.

In [4]:
src_branch = TIMESERIES_DIR / "branch.csv"
dst_branch = OUTPUT_DIR / "branch.csv"
shutil.copy2(src_branch, dst_branch)

branch_df = pd.read_csv(dst_branch)
print(f"Copied branch.csv: {len(branch_df)} branches")
print(f"From Bus range: {branch_df['From Bus'].min()}-{branch_df['From Bus'].max()}")
print(f"To Bus range: {branch_df['To Bus'].min()}-{branch_df['To Bus'].max()}")

Copied branch.csv: 255 branches
From Bus range: 2-123
To Bus range: 1-122


## Cell 5: Build timeseries_pointers.csv

Filter the base timeseries_pointers to only include generators that survived GTEP.

In [5]:
pointers_df = pd.read_csv(BASE_POINTERS)
print(f"Base timeseries_pointers: {len(pointers_df)} rows")

# Get invested gen set as strings
gen_uids_in_output = set(gen_df["GEN UID"].astype(str))

# Keep all non-Generator rows (Area/load pointers)
non_gen_mask = pointers_df["Category"] != "Generator"

# Keep Generator rows only for invested generators
gen_mask = (pointers_df["Category"] == "Generator") & (pointers_df["Object"].astype(str).isin(gen_uids_in_output))

filtered_pointers = pointers_df[non_gen_mask | gen_mask].copy()
print(f"Filtered timeseries_pointers: {len(filtered_pointers)} rows")
print(f"  Area rows: {(filtered_pointers['Category'] != 'Generator').sum()}")
print(f"  Generator rows: {(filtered_pointers['Category'] == 'Generator').sum()}")

# --- FIX: Add missing DAY_AHEAD pointers for candidate renewables ---
# The base timeseries_pointers.csv only has REAL_TIME pointers for candidates (pv_*-c, wind_*-c).
# Prescient needs DAY_AHEAD pointers too, otherwise the RUC solver uses scalar PMax MW
# and reporting.py:122 crashes with "TypeError: 'float' object is not subscriptable".
candidate_rt_ptrs = filtered_pointers[
    (filtered_pointers["Simulation"] == "REAL_TIME") &
    (filtered_pointers["Category"] == "Generator") &
    (filtered_pointers["Object"].astype(str).str.match(r"^(pv|wind)_"))
].copy()

# Check which candidates already have DAY_AHEAD pointers
existing_da = filtered_pointers[
    (filtered_pointers["Simulation"] == "DAY_AHEAD") &
    (filtered_pointers["Category"] == "Generator") &
    (filtered_pointers["Object"].astype(str).str.match(r"^(pv|wind)_"))
]

if len(candidate_rt_ptrs) > 0 and len(existing_da) == 0:
    da_ptrs = candidate_rt_ptrs.copy()
    da_ptrs["Simulation"] = "DAY_AHEAD"
    # Fix data file references: REAL_TIME_*.csv → DAY_AHEAD_*.csv
    da_ptrs["Data File"] = da_ptrs["Data File"].str.replace("REAL_TIME_", "DAY_AHEAD_")
    filtered_pointers = pd.concat([filtered_pointers, da_ptrs], ignore_index=True)
    print(f"  Added {len(da_ptrs)} DAY_AHEAD pointers for candidate renewables")

print(f"Final timeseries_pointers: {len(filtered_pointers)} rows")

# Write WITHOUT index
filtered_pointers.to_csv(OUTPUT_DIR / "timeseries_pointers.csv", index=False)
print(f"Wrote timeseries_pointers.csv")

Base timeseries_pointers: 1206 rows
Filtered timeseries_pointers: 640 rows
  Area rows: 246
  Generator rows: 394
  Added 122 DAY_AHEAD pointers for candidate renewables
Final timeseries_pointers: 762 rows
Wrote timeseries_pointers.csv


## Cell 6: Copy Static & Timeseries Files

Copy bus, simulation_objects, load, wind, and solar timeseries from the development directory.
Update simulation_objects.csv dates to 2035.

In [6]:
# Files to copy directly
direct_copy_files = [
    "bus.csv",
    "DAY_AHEAD_load.csv",
    "REAL_TIME_load.csv",
    "DAY_AHEAD_wind.csv",
    "REAL_TIME_wind.csv",
    "DAY_AHEAD_solar.csv",
    "REAL_TIME_solar.csv",
]

for fname in direct_copy_files:
    src = TIMESERIES_DIR / fname
    dst = OUTPUT_DIR / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied {fname}")
    else:
        print(f"WARNING: {fname} not found at {src}")

# simulation_objects.csv: update dates from 2019 to 2035 (4-digit year for clarity)
sim_obj = pd.read_csv(TIMESERIES_DIR / "simulation_objects.csv")
sim_obj.loc[sim_obj["Simulation_Parameters"] == "Date_From", "DAY_AHEAD"] = "1/1/2035 0:00"
sim_obj.loc[sim_obj["Simulation_Parameters"] == "Date_From", "REAL_TIME"] = "1/1/2035 0:00"
sim_obj.loc[sim_obj["Simulation_Parameters"] == "Date_To", "DAY_AHEAD"] = "12/31/2035 0:00"
sim_obj.loc[sim_obj["Simulation_Parameters"] == "Date_To", "REAL_TIME"] = "12/31/2035 0:00"
sim_obj.to_csv(OUTPUT_DIR / "simulation_objects.csv", index=False)
print("Wrote simulation_objects.csv (dates updated to 2035)")

print(f"\nAll files in {OUTPUT_DIR.name}/:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name:40s} {size_mb:8.2f} MB")

Copied bus.csv
Copied DAY_AHEAD_load.csv
Copied REAL_TIME_load.csv
Copied DAY_AHEAD_wind.csv
Copied REAL_TIME_wind.csv
Copied DAY_AHEAD_solar.csv
Copied REAL_TIME_solar.csv
Wrote simulation_objects.csv (dates updated to 2035)

All files in Prescient_2_2035/:
  DAY_AHEAD_load.csv                          12.23 MB
  DAY_AHEAD_solar.csv                          5.09 MB
  DAY_AHEAD_wind.csv                           7.66 MB
  REAL_TIME_load.csv                          12.89 MB
  REAL_TIME_solar.csv                          5.09 MB
  REAL_TIME_wind.csv                           7.66 MB
  branch.csv                                   0.02 MB
  bus.csv                                      0.01 MB
  gen.csv                                      0.05 MB
  simulation_objects.csv                       0.00 MB
  timeseries_pointers.csv                      0.04 MB


## Cell 7: Validation

Run comprehensive checks on the converted data.

In [7]:
warnings_list = []
errors_list = []

# 1. Required files check
required_files = [
    "gen.csv", "branch.csv", "bus.csv", "simulation_objects.csv",
    "timeseries_pointers.csv",
    "DAY_AHEAD_load.csv", "REAL_TIME_load.csv",
    "DAY_AHEAD_wind.csv", "REAL_TIME_wind.csv",
    "DAY_AHEAD_solar.csv", "REAL_TIME_solar.csv",
]
print("=== 1. Required Files ===")
for fname in required_files:
    exists = (OUTPUT_DIR / fname).exists()
    status = "OK" if exists else "MISSING"
    print(f"  {fname:40s} {status}")
    if not exists:
        errors_list.append(f"Missing required file: {fname}")

# 2. gen.csv integrity
print("\n=== 2. gen.csv Integrity ===")
gen_check = pd.read_csv(OUTPUT_DIR / "gen.csv")
gen_check["GEN UID"] = gen_check["GEN UID"].astype(str)

empty_uid = gen_check["GEN UID"].isna().sum()
print(f"  Empty GEN UID: {empty_uid}")
if empty_uid > 0:
    errors_list.append(f"{empty_uid} empty GEN UIDs")

# Bus ID must be integer
try:
    bus_ids = gen_check["Bus ID"].astype(int)
    print(f"  Bus ID: all integers, range {bus_ids.min()}-{bus_ids.max()}")
except (ValueError, TypeError) as e:
    errors_list.append(f"Bus ID contains non-integer values: {e}")
    print(f"  Bus ID: ERROR - non-integer values")

# PMax > 0
zero_pmax = (gen_check["PMax MW"] <= 0).sum()
print(f"  Generators with PMax MW <= 0: {zero_pmax}")
if zero_pmax > 0:
    bad = gen_check[gen_check["PMax MW"] <= 0]["GEN UID"].tolist()
    warnings_list.append(f"{zero_pmax} generators with PMax <= 0: {bad}")

# PMin >= 0
neg_pmin = (gen_check["PMin MW"] < 0).sum()
print(f"  Generators with PMin MW < 0: {neg_pmin}")
if neg_pmin > 0:
    errors_list.append(f"{neg_pmin} generators with negative PMin")

# 3. Pointer consistency
print("\n=== 3. Pointer Consistency ===")
ptr_check = pd.read_csv(OUTPUT_DIR / "timeseries_pointers.csv")
gen_ptrs = ptr_check[ptr_check["Category"] == "Generator"]
gen_uids_in_gen = set(gen_check["GEN UID"].astype(str))
gen_uids_in_ptrs = set(gen_ptrs["Object"].astype(str))
orphan_ptrs = gen_uids_in_ptrs - gen_uids_in_gen
print(f"  Generator pointers: {len(gen_ptrs)}")
print(f"  Generators in gen.csv: {len(gen_uids_in_gen)}")
print(f"  Orphan pointers (pointer without gen): {len(orphan_ptrs)}")
if orphan_ptrs:
    warnings_list.append(f"Orphan pointers: {sorted(orphan_ptrs)}")

# 4. Timeseries column consistency
print("\n=== 4. Timeseries Column Consistency ===")
for ts_file in ["DAY_AHEAD_wind.csv", "DAY_AHEAD_solar.csv"]:
    ts_df = pd.read_csv(OUTPUT_DIR / ts_file, nrows=1)
    ts_cols = set(str(c) for c in ts_df.columns) - {"Year", "Month", "Day", "Period"}
    # Check pointers reference columns that exist in timeseries
    ptrs_for_file = gen_ptrs[gen_ptrs["Data File"] == ts_file]
    ptr_objects = set(str(o) for o in ptrs_for_file["Object"])
    missing_cols = ptr_objects - ts_cols
    print(f"  {ts_file}: {len(ptr_objects)} pointers, {len(missing_cols)} missing columns")
    if missing_cols:
        warnings_list.append(f"{ts_file}: missing columns for {sorted(missing_cols)}")

# 5. Renewable PMax check
print("\n=== 5. Renewable PMax Check ===")
renewable_gens = gen_check[gen_check["Unit Type"].isin(["WIND", "PV"])]
zero_renew_pmax = renewable_gens[renewable_gens["PMax MW"] <= 0.01]
print(f"  Total renewable generators: {len(renewable_gens)}")
print(f"  With near-zero PMax: {len(zero_renew_pmax)}")
if len(zero_renew_pmax) > 0:
    warnings_list.append(f"{len(zero_renew_pmax)} renewables with near-zero PMax")

# 6. Cost curve check
print("\n=== 6. Cost Curve Check ===")
thermal_gens = gen_check[~gen_check["Unit Type"].isin(["WIND", "PV", "HYDRO"])]
zero_fp = thermal_gens[thermal_gens["Fuel Price $/MMBTU"] <= 0]
zero_hr = thermal_gens[thermal_gens["HR_incr_1"] <= 0]
print(f"  Thermal generators: {len(thermal_gens)}")
print(f"  With Fuel Price <= 0: {len(zero_fp)}")
print(f"  With HR_incr_1 <= 0: {len(zero_hr)}")
if len(zero_fp) > 0:
    errors_list.append(f"{len(zero_fp)} thermal gens with zero fuel price")
if len(zero_hr) > 0:
    errors_list.append(f"{len(zero_hr)} thermal gens with zero HR_incr_1")

# 7. Bus ID check
print("\n=== 7. Bus ID Check ===")
bus_check = pd.read_csv(OUTPUT_DIR / "bus.csv")
valid_bus_ids = set(bus_check["Bus ID"])
gen_bus_ids = set(gen_check["Bus ID"].astype(int))
invalid_buses = gen_bus_ids - valid_bus_ids
print(f"  Valid bus IDs: {len(valid_bus_ids)}")
print(f"  Generator bus IDs: {len(gen_bus_ids)}")
print(f"  Invalid bus IDs: {len(invalid_buses)}")
if invalid_buses:
    errors_list.append(f"Invalid bus IDs: {sorted(invalid_buses)}")

# 8. Summary
print("\n=== 8. Summary ===")
print(f"Generator counts by type:")
for ut, count in gen_check["Unit Type"].value_counts().items():
    total_mw = gen_check[gen_check["Unit Type"] == ut]["PMax MW"].sum()
    print(f"  {ut:8s}: {count:4d} generators, {total_mw:10.1f} MW")
print(f"  {'TOTAL':8s}: {len(gen_check):4d} generators, {gen_check['PMax MW'].sum():10.1f} MW")

print(f"\nErrors: {len(errors_list)}")
for e in errors_list:
    print(f"  ERROR: {e}")
print(f"Warnings: {len(warnings_list)}")
for w in warnings_list:
    print(f"  WARN: {w}")

validation_passed = len(errors_list) == 0
print(f"\nValidation: {'PASSED' if validation_passed else 'FAILED'}")

=== 1. Required Files ===
  gen.csv                                  OK
  branch.csv                               OK
  bus.csv                                  OK
  simulation_objects.csv                   OK
  timeseries_pointers.csv                  OK
  DAY_AHEAD_load.csv                       OK
  REAL_TIME_load.csv                       OK
  DAY_AHEAD_wind.csv                       OK
  REAL_TIME_wind.csv                       OK
  DAY_AHEAD_solar.csv                      OK
  REAL_TIME_solar.csv                      OK

=== 2. gen.csv Integrity ===
  Empty GEN UID: 0
  Bus ID: all integers, range 1-120
  Generators with PMax MW <= 0: 0
  Generators with PMin MW < 0: 0

=== 3. Pointer Consistency ===
  Generator pointers: 516
  Generators in gen.csv: 278
  Orphan pointers (pointer without gen): 0

=== 4. Timeseries Column Consistency ===
  DAY_AHEAD_wind.csv: 69 pointers, 0 missing columns
  DAY_AHEAD_solar.csv: 60 pointers, 0 missing columns

=== 5. Renewable PMax Check ===
  To

## Cell 8: Conversion Manifest

In [8]:
manifest = {
    "scenario": SCENARIO,
    "gtep_stage": GTEP_STAGE,
    "conversion_date": datetime.now().strftime("%Y-%m-%d"),
    "source_solution": "dispatchable_investments.json + renewable_investments.json",
    "source_gen": "Prescient_2/gen.csv (correct 4-segment cost curves) + fuel_cost3 from Prescient/gen.csv",
    "counts": {
        "total_generators": len(gen_df),
        "thermal_existing": len(gen_df[(~gen_df["Unit Type"].isin(["WIND", "PV", "HYDRO"])) & (~gen_df["GEN UID"].str.endswith("-c"))]),
        "thermal_candidate_new": len(gen_df[gen_df["GEN UID"].str.startswith("ct_fe_")]),
        "renewable_existing": len(gen_df[(gen_df["Unit Type"].isin(["WIND", "PV", "HYDRO"])) & (~gen_df["GEN UID"].str.endswith("-c"))]),
        "renewable_candidate_new": len(gen_df[(gen_df["Unit Type"].isin(["WIND", "PV"])) & (gen_df["GEN UID"].str.endswith("-c"))]),
        "retired_generators": len(retired_thermal),
        "branches": len(branch_df),
        "timeseries_pointer_rows": len(filtered_pointers),
    },
    "total_installed_renewable_mw": round(gen_df.loc[gen_df["Unit Type"].isin(["WIND", "PV", "HYDRO"]), "PMax MW"].sum(), 1),
    "fuel_price_source": "fuel_cost3 (2035 projection) — Fuel Price updated, HR_incr unchanged",
    "candidate_ct_hr_source": f"Median from existing CTs: HR_incr_1={ct_median_hr_incr_1:.2f}, HR_incr_2={ct_median_hr_incr_2:.2f}, HR_incr_3={ct_median_hr_incr_3:.2f}",
    "validation_passed": validation_passed,
    "errors": errors_list,
    "warnings": warnings_list,
}

manifest_path = OUTPUT_DIR / "conversion_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {manifest_path}")
print(json.dumps(manifest, indent=2))

Wrote /Users/yilu/Documents/GitHub/idaes-gtep/gtep/data/retirement_allowed_no_extreme_half_load_local/Prescient_2_2035/conversion_manifest.json
{
  "scenario": "retirement_allowed_no_extreme_half_load_local",
  "gtep_stage": 3,
  "conversion_date": "2026-04-10",
  "source_solution": "dispatchable_investments.json + renewable_investments.json",
  "source_gen": "Prescient_2/gen.csv (correct 4-segment cost curves) + fuel_cost3 from Prescient/gen.csv",
  "counts": {
    "total_generators": 278,
    "thermal_existing": 127,
    "thermal_candidate_new": 22,
    "renewable_existing": 68,
    "renewable_candidate_new": 61,
    "retired_generators": 0,
    "branches": 255,
    "timeseries_pointer_rows": 762
  },
  "total_installed_renewable_mw": 11268.1,
  "fuel_price_source": "fuel_cost3 (2035 projection) \u2014 Fuel Price updated, HR_incr unchanged",
  "candidate_ct_hr_source": "Median from existing CTs: HR_incr_1=1360.40, HR_incr_2=1381.58, HR_incr_3=1402.77",
  "validation_passed": true,
  